# Assignment 4

Name: Aziz Sayyad
Roll: 381069
PRN: 22420090
Batch: A3

#### Build a Named Entity Recognition (NER) system for extracting entities from real-world text such as news articles or social media data. And measure its accuracy, precision, recall, and F1\ score. 

In [1]:
# Install dependencies (transformers + metrics + CPU torch)
%pip -q install transformers seqeval
%pip -q install torch --index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import math
from typing import List, Dict, Tuple
from transformers import pipeline

# Load pretrained NER pipeline (aggregated spans for cleaner entities)
ner_tagger = pipeline(
    task="token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device="cpu",
)

print("Loaded model:", ner_tagger.model.name_or_path)


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

c:\Users\Aziz\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aziz\.cache\huggingface\hub\models--dslim--bert-base-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


Loaded model: dslim/bert-base-NER


In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Span:
    start: int
    end: int
    label: str


def mentions_to_spans(text: str, mentions: List[Tuple[str, str]]) -> List[Span]:
    """Convert ordered mention strings to character spans using sequential search."""
    spans: List[Span] = []
    cursor = 0
    for mention, label in mentions:
        start = text.index(mention, cursor)
        end = start + len(mention)
        spans.append(Span(start, end, label))
        cursor = end
    return spans


raw_samples = [
    (
        "On September 12, 2023, Apple Inc. unveiled a new iPhone at its annual event in Cupertino.",
        [("September 12, 2023", "DATE"), ("Apple Inc.", "ORG"), ("Cupertino", "GPE")],
    ),
    (
        "In Washington, Elon Musk had a meeting with NASA representatives regarding the Artemis program.",
        [("Washington", "GPE"), ("Elon Musk", "PERSON"), ("NASA", "ORG"), ("Artemis", "EVENT")],
    ),
    (
        "After the earthquake in Turkey, the UN organized an emergency meeting in New York.",
        [("Turkey", "GPE"), ("UN", "ORG"), ("New York", "GPE")],
    ),
    (
        "On May 15, 2022, Chelsea was defeated 2-0 by Manchester United at Wembley.",
        [("May 15, 2022", "DATE"), ("Chelsea", "ORG"), ("Manchester United", "ORG"), ("Wembley", "GPE")],
    ),
    (
        "During the Climate Summit 2021 held in Paris, Barack Obama delivered a speech.",
        [("Climate Summit 2021", "EVENT"), ("Paris", "GPE"), ("Barack Obama", "PERSON")],
    ),
    (
        "On March 11, 2020, the World Health Organization officially declared COVID-19 a global pandemic.",
        [("March 11, 2020", "DATE"), ("World Health Organization", "ORG"), ("COVID-19", "EVENT")],
    ),
    (
        "To expand its services across India, Amazon launched a new data center in Mumbai.",
        [("India", "GPE"), ("Amazon", "ORG"), ("Mumbai", "GPE")],
    ),
        (
        "In June 2023, Cristiano Ronaldo signed a new contract with Al Nassr FC.",
        [("June 2023", "DATE"), ("Cristiano Ronaldo", "PERSON"), ("Al Nassr FC", "ORG")],
    ),
]


dataset = [
    {
        "text": text,
        "gold": mentions_to_spans(text, mentions),
    }
    for text, mentions in raw_samples
]

len(dataset)


8

In [3]:
import pandas as pd

sample_text = dataset[0]["text"]
preds = ner_tagger(sample_text)
pd.DataFrame(preds)[["word", "entity_group", "start", "end", "score"]]


,word,entity_group,start,end,score
0,Apple Inc,ORG,23,32,0.999399
1,iPhone,MISC,49,55,0.993785
2,Cupertino,LOC,79,88,0.995020


In [4]:
def evaluate(samples: List[Dict]) -> Dict[str, float]:
    tp = fp = fn = 0
    for sample in samples:
        text = sample["text"]
        gold_spans = { (s.start, s.end, s.label) for s in sample["gold"] }
        pred_raw = ner_tagger(text)
        pred_spans = { (p["start"], p["end"], p["entity_group"]) for p in pred_raw }

        tp += len(gold_spans & pred_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = tp / (tp + fp + fn) if tp + fp + fn else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "entity_accuracy": accuracy,
    }


metrics = evaluate(dataset)
metrics


{'tp': 7,
 'fp': 18,
 'fn': 19,
 'precision': 0.28,
 'recall': 0.2692307692307692,
 'f1': 0.27450980392156865,
 'entity_accuracy': 0.1590909090909091}